In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/baseline.pt
/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/part_A.pt
/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/part_B.pt


In [2]:
!find /kaggle/input -name "*.pt"

/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/baseline.pt
/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/part_A.pt
/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/part_B.pt


In [3]:
!pip install -q ultralytics openvino nncf onnx onnxslim onnxruntime

In [4]:
BASELINE_PT = "/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/baseline.pt"   # YOLO11n trained at 640
RUNB_PT     = "/kaggle/input/datasets/suddapallikishore/vsdrone-trained-weights/part_B.pt"       # YOLO11n trained at 1280

In [5]:
import shutil, random
from pathlib import Path
import numpy as np, pandas as pd
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

OUT = Path("/kaggle/working/optimise")
OUT.mkdir(exist_ok=True)

data = check_det_dataset("VisDrone.yaml")
test_imgs = sorted(Path(data["test"]).glob("*.jpg"))
random.seed(1)
bench_imgs = random.sample(test_imgs, 45)

def size_mb(path):
    p = Path(path)
    files = p.rglob("*") if p.is_dir() else [p]
    return sum(f.stat().st_size for f in files if f.is_file()) / 1e6

def cpu_speed(path, imgsz, images, warmup=5):
    model = YOLO(str(path), task="detect")
    for p in images[:warmup]:
        model.predict(str(p), imgsz=imgsz, device="cpu", verbose=False)
    times = []
    for p in images[warmup:]:
        r = model.predict(str(p), imgsz=imgsz, device="cpu", verbose=False)[0]
        times.append(sum(r.speed.values()))
    return float(np.median(times)), float(np.percentile(times, 95))

def test_accuracy(path, imgsz):
    m = YOLO(str(path), task="detect").val(
        data="VisDrone.yaml", split="test", imgsz=imgsz, batch=1,
        device="cpu", plots=False, verbose=False)
    return m.box.map50, m.box.map, m.box.mr

MODELS = {"Baseline n640": (BASELINE_PT, 640), "Run B n1280": (RUNB_PT, 1280)}
rows = []

for name, (src, size) in MODELS.items():
    tag = name.replace(" ", "_")
    pt32, pt8 = OUT / f"{tag}_fp32.pt", OUT / f"{tag}_int8.pt"
    shutil.copy(src, pt32)
    shutil.copy(src, pt8)

    versions = {
        "PyTorch FP32":  lambda: str(pt32),
        "ONNX FP32":     lambda: YOLO(str(pt32)).export(format="onnx", imgsz=size),
        "OpenVINO FP32": lambda: YOLO(str(pt32)).export(format="openvino", imgsz=size),
        "OpenVINO INT8": lambda: YOLO(str(pt8)).export(format="openvino", imgsz=size,
                                                       quantize=8, data="VisDrone.yaml", fraction=0.1),
    }

    for vname, make in versions.items():
        row = {"model": name, "version": vname}
        try:
            path = make()
            med, p95 = cpu_speed(path, size, bench_imgs)
            row.update({"size MB": round(size_mb(path), 1),
                        "CPU ms (median)": round(med, 1),
                        "CPU ms (p95)": round(p95, 1),
                        "CPU FPS": round(1000 / med, 1)})
            if name == "Run B n1280":
                m50, m95, rec = test_accuracy(path, size)
                row.update({"test mAP50": round(m50, 3),
                            "test mAP50-95": round(m95, 3),
                            "test recall": round(rec, 3)})
        except Exception as e:
            row["error"] = f"{type(e).__name__}: {e}"[:200]
        rows.append(row)
        print(row)

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/optimisation_results.csv", index=False)
print("\n===== FINAL RESULTS =====")
print(df.to_string(index=False))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

WARNING ⚠️ Dataset 'VisDrone.yaml' images not found, missing path '/kaggle/working/datasets/VisDrone/images/val'
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-test-dev.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.1Kfiles/s 2.9s
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-val.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 971.0files/s 1.1s
Unzipping /kaggle/working/datasets/VisDrone/VisDrone2019-DET-train.zip to /kaggle/working/datasets/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 2.4Kfiles/s 5.5s
Converting train: ━━━━━━━━━━━━ 6471 2.5Kit/s 2.6s

Output()

Output()

OpenVINO: export success ✅ 19.2s, saved as '/kaggle/working/optimise/Baseline_n640_int8_int8_openvino_model/' (3.2 MB)

Export complete (19.5s)
Results saved to /kaggle/working/optimise/Baseline_n640_int8_int8_openvino_model
Predict:         yolo predict task=detect model=/kaggle/working/optimise/Baseline_n640_int8_int8_openvino_model/ imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/optimise/Baseline_n640_int8_int8_openvino_model/ imgsz=640 data=/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml 
Visualize:       https://netron.app
Loading /kaggle/working/optimise/Baseline_n640_int8_int8_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...
{'model': 'Baseline n640', 'version': 'OpenVINO INT8', 'size MB': 3.4, 'CPU ms (median)': 41.0, 'CPU ms (p95)': 48.9, 'CPU FPS': 24.4}
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 l

Output()

Output()

OpenVINO: export success ✅ 39.2s, saved as '/kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model/' (3.5 MB)

Export complete (39.8s)
Results saved to /kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model
Predict:         yolo predict task=detect model=/kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model/ imgsz=1280 
Validate:        yolo val task=detect model=/kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model/ imgsz=1280 data=/usr/local/lib/python3.13/dist-packages/ultralytics/cfg/datasets/VisDrone.yaml 
Visualize:       https://netron.app
Loading /kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...
Loading /kaggle/working/optimise/Run_B_n1280_int8_int8_openvino_model/ for OpenVINO inference...
Using OpenVINO LATENCY mode for batch=1 inference on CPU...
Setting batch=1 input of shape (1, 3, 1280, 1280)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 127